# Model training and evaluation

Train and evaluate the PM2.5 model (Phases I-J). A Random Forest baseline is compared with a tuned XGBoost model, always scored on the spatial folds defined in notebook 08, with the random folds kept as the optimistic reference. Headline numbers come from pooled out-of-fold predictions on the µg/m³ scale.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor

import monitoring_deserts.models.spatial_cv as spatial_cv

## Configuration

In [11]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

CV_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "cross_validation" # outputs of notebook 08
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
MODELS_DIR = Path(gen_config["data_dir"]["models"])
PLOTS_DIR = Path(gen_config["output_dir"]["plots"]) / "model_training"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validation settings, as saved by notebook 08
with open("../configs/cross_validation_metadata.yaml") as f:
    cv_config = yaml.safe_load(f)

RANDOM_STATE = cv_config["random_state"]
N_FOLDS = cv_config["n_folds"]

# Station-month table from notebook 7 (cleaned, with the log-transformed target)
analysis_df = pd.read_parquet(PROCESSED_DATA_DIR / "master_station_df.parquet")

# The parquet file loads month_of_year back as a plain integer, so restore the categorical
# dtype notebook 7 gave it (XGBoost only treats it as categorical with enable_categorical=True)
analysis_df["month_of_year"] = pd.Categorical(analysis_df["month_of_year"], categories=range(1, 13))

# Reuse the exact blocks and folds of notebook 08 (re-clustering could give different blocks)
station_folds = pd.read_parquet(CV_DATA_DIR / "station_blocks_folds.parquet")
analysis_df = analysis_df.merge(station_folds[["station_id", "spatial_block_id", "spatial_fold"]],
                                on="station_id", how="left", validate="many_to_one")
assert analysis_df["spatial_fold"].notna().all(), "stations missing from the saved fold table"

gkf_splits = [(np.where(analysis_df["spatial_fold"] != k)[0], np.where(analysis_df["spatial_fold"] == k)[0])
              for k in range(N_FOLDS)]
random_splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(analysis_df))

## Step 1 - Choose features

In [4]:
# Cyclic encoding the month_of_year for random forest
analysis_df["sin_month"] = np.sin(2 * np.pi * analysis_df["month_of_year"].astype(int) / 12)
analysis_df["cos_month"] = np.cos(2 * np.pi * analysis_df["month_of_year"].astype(int) / 12)

rfr_feature_cols = ["no2", "aai",
                    "total_precipitation", "temperature", "dew_point",
                    "u_wind", "v_wind", "wind_magnitude",
                    "boundary_layer_height", "surface_pressure", "relative_humidity",
                    "sin_month", "cos_month",
                   ]
xgb_feature_cols = ["no2", "aai",
                    "total_precipitation", "temperature", "dew_point",
                    "u_wind", "v_wind", "wind_magnitude",
                    "boundary_layer_height", "surface_pressure", "relative_humidity",
                    "month_of_year",
                   ]
target_col = "log1p_pm25"

rfr_X = analysis_df[rfr_feature_cols]
xgb_X = analysis_df[xgb_feature_cols]
y = analysis_df[target_col]

## Step 2 - Random forest baseline with Group K-Fold splits

In [5]:
#rfr = RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)
#param_distributions = {"n_estimators" : [400, 500, 600, 700],
#                       "max_depth" : [5, 7, 10, 15, 20],
#                       "min_samples_leaf" : [5, 10, 15, 20, 25],
#                       "max_features" : [.2, .5, .7, .9],
#                      }
#baseline_search = RandomizedSearchCV(rfr, param_distributions, scoring="neg_root_mean_squared_error", 
#                                     n_jobs=-1, n_iter=20, cv=gkf_splits, 
#                                     random_state=RANDOM_STATE, verbose=1)
#baseline_search.fit(rfr_X, y);
#print(f"Best Random Forest estimator: {baseline_search.best_estimator_}")
#print(f"Lowest RMSE score: {- baseline_search.best_score_:.3f}")

Fitting 8 folds for each of 20 candidates, totalling 160 fits


In [15]:
## Save the fitted model and the results of the CV search
#joblib.dump(baseline_search.best_estimator_, MODELS_DIR / "random_forest.joblib")
#joblib.dump(baseline_search, MODELS_DIR / "baseline_search.joblib")

# Load the fitted model and search results
rfr_best = joblib.load(MODELS_DIR / "random_forest.joblib")
baseline_search = joblib.load(MODELS_DIR / "baseline_search.joblib")

print(f"Best Random Forest estimator: {baseline_search.best_estimator_}")
print(f"Lowest RMSE score: {- baseline_search.best_score_:.3f}")

Best Random Forest estimator: RandomForestRegressor(max_depth=7, max_features=0.2, min_samples_leaf=10,
                      n_estimators=400, n_jobs=-1, random_state=42)
Lowest RMSE score: 0.462


## Step 3 - Tune XGBoost

In [13]:
xgbr = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, enable_categorical=True,)
param_distributions = {"n_estimators" : [100 * x for x in range(1, 11)],
                       "max_leaves" : [x for x in range(21)],
                       "learning_rate" : [.01, .05, .1, .15],
                       "subsample" : [.2, .4, .5, .7, 1.],
                      }
xgb_search = RandomizedSearchCV(xgbr, param_distributions, scoring="neg_root_mean_squared_error", 
                                n_jobs=-1, n_iter=20, cv=gkf_splits, 
                                random_state=RANDOM_STATE, verbose=1)
xgb_search.fit(xgb_X, y);

Fitting 8 folds for each of 20 candidates, totalling 160 fits


In [17]:
## Save the fitted model and the results of the CV search
#joblib.dump(xgb_search.best_estimator_, MODELS_DIR / "xgb_best.joblib")
#joblib.dump(xgb_search, MODELS_DIR / "xgb_search.joblib")

# Load the fitted model and search results
xgb_best = joblib.load(MODELS_DIR / "xgb_best.joblib")
xgb_search = joblib.load(MODELS_DIR / "xgb_search.joblib")

print(f"Best XGBRegressor estimator: {xgb_search.best_estimator_}")
print(f"Lowest RMSE score: {- xgb_search.best_score_:.3f}")

Best XGBRegressor estimator: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=17, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)
Lowest RMSE score: 0.467


In [26]:
# Check if I am on a plateau or if it makes sense to increase iterations
tmp = (pd.DataFrame(xgb_search.cv_results_)
       .sort_values(by="rank_test_score", ascending=False)[["mean_test_score", "std_test_score"]]
       .reset_index(drop=True)
      )
tmp["pos_score"] = - tmp["mean_test_score"]
tmp["n_std_from_better_score"] = (- tmp["pos_score"] + tmp["pos_score"].shift(1)) / tmp["std_test_score"]
print(tmp.head(5))
# Seems like I am on a plateau. Keeping n_iter = 20                              

   mean_test_score  std_test_score  pos_score  n_std_from_better_score
0        -0.538239        0.074367   0.538239                      NaN
1        -0.526756        0.077395   0.526756                 0.148370
2        -0.521554        0.070760   0.521554                 0.073516
3        -0.521131        0.072002   0.521131                 0.005883
4        -0.516565        0.096443   0.516565                 0.047345


## Outline

1. ~~**Step 1 - Features and target.** Choose the predictor columns and the target (`log1p_pm25`, back-transformed with `np.expm1` for every quoted number). Latitude and longitude stay out of the features, otherwise the model can memorise stations instead of learning the pollution signal. The month enters as `sin_month`/`cos_month` for the Random Forest and as a categorical `month_of_year` for XGBoost.~~
2. ~~**Step 2 - Random Forest baseline.** Tune a Random Forest with `RandomizedSearchCV`, passing `gkf_splits` as `cv` and scoring with `neg_root_mean_squared_error`. This is the reference the XGBoost model has to beat on the spatial folds.~~
3. ~~**Step 3 - Tune XGBoost.** Run `RandomizedSearchCV` over `n_estimators`, `max_leaves`, `learning_rate` and `subsample` with the same folds, scoring and number of iterations, and `enable_categorical=True` for `month_of_year`. The spread of the candidate scores shows the search is on a plateau, so `n_iter` stays at 20.~~
4. **Step 4 - Fold-by-fold comparison of the tuned models.** The mean search scores (RF 0.462, XGBoost 0.467 RMSE on the log scale) differ by far less than the fold-to-fold spread, so the means cannot decide between the models. Run the `best_estimator_` of each search, with its own feature matrix, through `spatial_cv.run_cross_validation` on `gkf_splits`, and compare the per-fold RMSE: the paired difference on each fold, and on how many folds each model wins. Apply the decision rule of plan v14 (Phase I step 3): a gap well inside the fold spread is a tie and the simpler Random Forest stays the definitive model. Both searches looked at these folds, so both scores are slightly optimistic in the same way; note this next to the numbers.
5. **Step 5 - Headline metrics.** For the chosen model, compute R², RMSE and MAE on the µg/m³ scale from the pooled out-of-fold predictions (not the average of per-fold scores, which weights small folds too heavily). Keep the per-fold values as a diagnostic of how much the score varies from region to region.
6. **Step 6 - Predicted vs observed.** Scatter plot of every held-out station-month, coloured by spatial block, with the 1:1 line. It shows whether errors come from a few regions or whole blocks, and whether the model compresses the high PM2.5 values.
7. **Step 7 - Feature importance.** Built-in importances from the Random Forest and XGBoost, plus SHAP values if time allows, to explain which signals drive the predictions.
8. **Step 8 - Random vs spatial comparison.** Score the chosen model on `random_splits` as well (same `run_cross_validation`) and put the random-CV and spatial-CV scores side by side, in text and in a small table. The gap is the methodological finding of Phase H.
9. **Step 9 - Save.** Write `configs/model_training_metadata.yaml` (features, search spaces, best parameters, headline metrics) and the out-of-fold predictions, and decide where the fitted model is stored.